In [1]:
!pip install -q -U "trl==0.8.6" peft bitsandbytes transformers torch datasets evaluate rouge_score

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 113.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 12.2 MB/s eta 0:00:00
   ━━

In [ ]:
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import evaluate
from google.colab import drive
from tqdm import tqdm

# 1. Setup Environment
drive.mount('/content/drive')

# Paths & Token
ADAPTER_PATH = "/content/drive/MyDrive/AidataMiningProject/Models/gemma_2b_finetuned"
DATA_PATH = "/content/drive/MyDrive/AidataMiningProject/datasets/mergedDataset.csv"
HF_TOKEN = "hf_xxxxxxxxxxx"  # Replace with your actual token

# 2. Load Base Model (Gemma 2B) in 4-bit
model_name = "google/gemma-2b-it"
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False,
)

print("Loading Base Model...")
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN
)

tokenizer = AutoTokenizer.from_pretrained(model_name, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

# 3. Load Fine-Tuned Adapters
print("Loading Adapter Weights...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval() # Set to evaluation mode

# 4. Prepare Test Data
# Load dataset and take a random sample of 50 for testing
df = pd.read_csv(DATA_PATH)[["article_text", "summary"]].dropna()
test_df = df.sample(n=50, random_state=42).reset_index(drop=True)

print(f"Starting evaluation on {len(test_df)} samples...")

# 5. Inference Function
def generate_summary(text):
    prompt = f"""### Instruction:
Summarize the following Turkish text concisely.

### Text:
{text}

### Summary:
"""
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,    # Deterministic output for evaluation
            temperature=0.0,
            repetition_penalty=1.2
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract summary part
    try:
        summary = full_output.split("### Summary:\n")[1].strip()
    except:
        summary = full_output

    return summary

# 6. Generate Summaries Loop
generated_summaries = []
reference_summaries = test_df['summary'].tolist()

# Use tqdm for progress bar
for article in tqdm(test_df['article_text']):
    summary = generate_summary(article)
    generated_summaries.append(summary)

# 7. Compute ROUGE Scores
rouge = evaluate.load('rouge')
results = rouge.compute(predictions=generated_summaries, references=reference_summaries)

# 8. Display Results
print("\n" + "="*30)
print("📊 ROUGE SCORE RESULTS")
print("="*30)
print(f"ROUGE-1 (Unigram): {results['rouge1']:.4f}")
print(f"ROUGE-2 (Bigram):  {results['rouge2']:.4f}")
print(f"ROUGE-L (Longest): {results['rougeL']:.4f}")
print("="*30)

# Show an example
print("\n📝 Example Comparison:")
print(f"Original:  {reference_summaries[0]}")
print(f"Generated: {generated_summaries[0]}")

Mounted at /content/drive
Loading Base Model...


config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

Loading Adapter Weights...
Starting evaluation on 50 samples...


100%|██████████| 50/50 [04:19<00:00,  5.18s/it]



📊 ROUGE SCORE RESULTS
ROUGE-1 (Unigram): 0.2045
ROUGE-2 (Bigram):  0.1019
ROUGE-L (Longest): 0.1659

📝 Example Comparison:
Original:  Zimbabve, bu yıl mısır üretiminde görülen artışın ardından bu ürünün ithalatını yasakladı.
Generated: Afrika Devletleri Merkezi (ADB), ülkenin iç tarımında önemli rol oynayan Mısır’a yönelik fiyat kontrolü uygulamasının sona erdiğini duyurdu. ABD Başkanı Donald Trump tarafından yürürlüğe giren "Mısır İhracatına Yönelik Uygulamalar" kararnamesi kapsamında uygulamaya başlandı. Bu kararla yurt dışındaki ihracatçılar için satışları durduruldu.


ABD Tarım Bakanlığına
